# Fine-tuning ModernBERT with LoRA: Signpost

**Part 4 of "LLMs the Hard Way"** (branch `signpost`). `02_frozen_router.ipynb` routed conversations well with a **frozen** encoder, but grounded questions about a text plateaued at 114 / 170. No head or amount of data could make frozen features *read*. Here the encoder itself learns: hand-written **LoRA** adapters on every attention and MLP layer of ModernBERT-large (7.2 M trained parameters next to 395 M frozen), plus a small scoring head.

**One model, two jobs**, in one question format, with one forward pass per option:
- **agent routing** with chat history and the current agent: "Which agent should handle the user's latest message?";
- **grounded questions** about a text: yes / no, one of N options, or a level on an ordered scale.

```text
<role>: <msg> [SEP] … [SEP] current agent: X [SEP] question: <q> [SEP] candidate: <option>. <description>
        -> ModernBERT-large (+ LoRA) -> mean pool -> head -> score;   softmax over the options -> answer
```

This notebook trains that model and tests it. It ends by exporting it as **Signpost**, published at [huggingface.co/skamalj/signpost](https://huggingface.co/skamalj/signpost) (tag `signpost-v1.0` marks the exact state that produced it).

**Tests** (never used for training), the same items as in `02` and `04_decider_benchmark.ipynb`:

| Test | What it contains | Items |
|---|---|---|
| E10 | Realistic multi-turn chats at 13 companies, routed live (the model's own previous pick is the current agent) | 193 turns; 83 at the 6 test companies |
| E11 | One opening message, 5 agents | 100 |
| Banking | Banking customer messages, 5 banking agents | 50 · 225 |
| E12 | Grounded questions; the answer is always in the text | 170 (58 yes / no, 59 choice, 53 score) |
| Independent | Written by a separate author from a brief, with their own businesses: questions, live routing, first messages | 300 · 172 turns · 200 |

**Latency** is measured on every test: one question at a time, warm, tokenization included, p50 / p95 / mean in ms.

**Compared with** the frozen system (`02`, run 17) and the 2B [Strands decider](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19) (`04`).

## 1 — Setup

All settings are environment variables; the defaults train and export Signpost (run 25).

| Setting | Default | Meaning |
|---|---|---|
| `RUN_ENCODERS` | `modernbert-large` | encoders to train, in order (`modernbert-large`, `nemotron-1b`; `modernbert-base` for a smoke check) |
| `TRAIN_MODE` | `combined` | `combined`: one model for routing and questions; `separate`: two specialists, one per job |
| `SCORE_LOSSES` | `ordinal` | `plain` (cross-entropy on the true level) and / or `ordinal` (order-aware); each listed variant is trained in turn |
| `EPOCHS` | 2 | passes over all training questions |
| `LORA_R`, `LORA_ALPHA` | 16, 32 | LoRA rank and scale |
| `LR`, `HEAD_LR` | 2e-4, 1e-3 | learning rates of the adapters and of the head |
| `BATCH_ROUTE`, `BATCH_CHOICE` | 4, 16 | questions per step (a routing question brings 5 longer inputs, a choice question 2–5 short ones) |
| `MAX_LEN` | 320 | tokens per input; the longest routing input is about 255. If anything is cut, it is the start of the history, never the candidate |
| `SEED` | 42 | the first seed; the collapse guard moves on by +1000 per retry |
| `EXPORT` | 1 | after training, merge LoRA, save `signpost/` + `signpost.zip`, and verify the reload |
| `LORA_SMOKE` | 0 | 1 = a tiny code check on a CPU |

**The experiments of runs 19–25 are settings of this notebook** (with the data of each run's own commit; this branch has the final data):

| Run | Question | Settings |
|---|---|---|
| 19 | One model for both jobs; ModernBERT-large vs Nemotron-1B | `RUN_ENCODERS=modernbert-large,nemotron-1b SCORE_LOSSES=plain` |
| 21 | Plain vs order-aware score loss | `SCORE_LOSSES=plain,ordinal` |
| 23 | Nemotron-1B on the final recipe | `RUN_ENCODERS=nemotron-1b EPOCHS=3 EXPORT=0` |
| 24 | Two specialists vs one combined model | `TRAIN_MODE=separate EXPORT=0` |
| 25 | **Signpost**: train and export | the defaults |

In [ ]:
!pip install -q "transformers==5.17.0"
import copy, gc, json, math, os, random, statistics, subprocess, time
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE = os.environ.get("LORA_SMOKE") == "1"
RUN_ENCODERS = os.environ.get("RUN_ENCODERS", "modernbert-base" if SMOKE else "modernbert-large").split(",")
# "combined" trains one model on routing and questions together (Signpost); "separate" trains two independent models from
# scratch, one on routing only and one on the questions only, each scored on its own tests (run 24).
TRAIN_MODE = os.environ.get("TRAIN_MODE", "combined")
# Export the trained combined model as a self-contained "signpost/" folder (+ signpost.zip) and verify it.
EXPORT = os.environ.get("EXPORT", "1") == "1"
LORA_R, LORA_ALPHA, LORA_DROPOUT = int(os.environ.get("LORA_R", "16")), float(os.environ.get("LORA_ALPHA", "32")), 0.1
EPOCHS = int(os.environ.get("EPOCHS", "1" if SMOKE else "2"))
LR, HEAD_LR = float(os.environ.get("LR", "2e-4")), float(os.environ.get("HEAD_LR", "1e-3"))
BATCH_ROUTE, BATCH_CHOICE = int(os.environ.get("BATCH_ROUTE", "4")), int(os.environ.get("BATCH_CHOICE", "16"))
MAX_LEN = int(os.environ.get("MAX_LEN", "320"))
# The score-question loss. "plain" = cross-entropy on the true level; "ordinal" = SCORE_EPS of the target spread onto the
# neighbouring levels + SCORE_DIST_W x |expected level - true level| (adopted in run 21). Each listed variant is trained in turn.
SCORE_LOSSES = os.environ.get("SCORE_LOSSES", "ordinal").split(",")
SCORE_EPS, SCORE_DIST_W = float(os.environ.get("SCORE_EPS", "0.1")), float(os.environ.get("SCORE_DIST_W", "0.5"))
SEED = int(os.environ.get("SEED", "42"))
K_MAX = 5
ROUTE_Q = "Which agent should handle the user's latest message?"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path("lora_signpost_run.log")


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


import transformers
log(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(DEVICE)})" if AMP else "") + f" | encoders {RUN_ENCODERS}" + (" | SMOKE CHECK" if SMOKE else ""))

## 2 — Data: one question format for routing and questions

Every item becomes `{task, messages, current, question, options {name: description}, answer}`:
- **routing:** the training conversations (templates + realistic) with their 5 domain agents (name + description) and the agent currently holding the chat;
- **questions:** the grounded yes / no, choice and score items. The text is a single user message, there is no current agent, and the options have no description. For score questions, the options are the levels from low to high.

**Dev sets** (early stopping and the collapse guard only): 10% of the routing **domains** (unseen companies, as in the tests) and 10% of the question **texts**. **Tests:** E10, E11, banking, E12 and the independent set, exactly as in `02_frozen_router.ipynb` and `04_decider_benchmark.ipynb`.

**Independence.** The training data was generated by the builders in `data/modernbert_router/`, with guards against near-copies of E12 (5-gram and content-word overlap). The independent set is a final check only: no training data was added or changed because of it.

In [ ]:
DATA_BRANCH = "signpost"                                    # this notebook's branch (cloned on Kaggle / Colab)
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"
load = lambda p: json.load(open(DATA_DIR / p))

# ---- routing training data
domains = load("generic/domains.json"); domains.update(load("realistic_train/domains.json"))
route_convs = load("generic/conversations.json") + load("realistic_train/conversations.json")
route_items = [{"task": "routing", "domain": ex["domain"], "messages": ex["messages"], "current": ex["current_agent"],
                "question": ROUTE_Q, "options": domains[ex["domain"]]["agents"], "answer": ex["target_agent"]} for ex in route_convs]
# ---- choice training data
choice_items = [{"task": it["type"], "domain": it["domain"], "messages": [{"role": "user", "content": it["state"]}], "current": None,
                 "question": it["question"], "options": {o: "" for o in it["options"]}, "answer": it["answer"], "state": it["state"],
                 "levels": list(it["options"])}            # for score questions: the levels from low to high
                for it in load("choice_train/items.json")["items"]]

rng = random.Random(SEED)
route_domains = sorted({x["domain"] for x in route_items}); rng.shuffle(route_domains)
dev_domains = set(route_domains[: max(1, len(route_domains) // 10)])
texts = sorted({x["state"] for x in choice_items}); rng.shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])
train_route = [x for x in route_items if x["domain"] not in dev_domains]
dev_route = [x for x in route_items if x["domain"] in dev_domains]
train_choice = [x for x in choice_items if x["state"] not in dev_texts]
dev_choice = [x for x in choice_items if x["state"] in dev_texts]

# ---- tests
real_convs = load("realistic/conversations.json")
ft = load("first_turn/messages.json")
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
BANKING_AGENTS = {                                          # exactly as in 02_frozen_router.ipynb, Cell 37
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
# ---- independent test set (written outside the project; see experiments/independent_test_brief.md)
ind_q = load("independent/questions.json")["items"]
ind_r = load("independent/routing.json")
ind_f = load("independent/first_turn.json")["messages"]
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])           # the same 6 test companies as 02_frozen_router.ipynb, Cell 70

if SMOKE:
    train_route, dev_route, train_choice, dev_choice = train_route[::400], dev_route[::100], train_choice[::400], dev_choice[::100]
    real_convs, banking, e12 = real_convs[:2], banking[:6], e12[::20]
    ft = {"companies": ft["companies"], "messages": ft["messages"][::20]}

for name, xs in [("train routing", train_route), ("train choice", train_choice), ("dev routing", dev_route), ("dev choice", dev_choice)]:
    print(f"{name:14s}: {len(xs):5d} questions | " + ", ".join(f"{t} {n}" for t, n in Counter(x["task"] for x in xs).items()))
print(f"dev routing domains (unseen in training): {sorted(dev_domains)}")
print(f"independent: questions {len(ind_q)} | routing {len(ind_r)} conversations / {sum(len(c['user_turns']) for c in ind_r)} turns "
      f"(flagged: {dict(Counter(u['flag'] for c in ind_r for u in c['user_turns'] if u['flag']))}) | first turn {len(ind_f)}")
print(f"tests: E10 {len(real_convs)} conversations / {sum(len(c['user_turns']) for c in real_convs)} turns | E11 {len(ft['messages'])} | "
      f"banking {len(banking)} | E12 {len(e12)}")

## 3 — Encoder with LoRA, scoring head, input format

**LoRA, written by hand.** Every chosen linear layer `W` of the frozen encoder is wrapped as `y = W x + (alpha / r) · B (A x)`. `A` is `[r, in]`, `B` is `[out, r]` and starts at **zero**, so training starts from the unchanged encoder. ModernBERT: `Wqkv`, `Wo`, `Wi` (attention and MLP, all 28 layers: 112 layers, 7.2 M parameters). Nemotron (Llama): `q/k/v/o_proj` and `gate/up/down_proj`, with **gradient checkpointing** to fit the T4.

`LoRALinear`, `add_lora` and `merge_lora` (Section 4) are about 30 lines in total. With the `peft` library, the same setup would be:

```python
from peft import LoraConfig, get_peft_model
enc = get_peft_model(enc, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["Wqkv", "Wo", "Wi"]))
...
enc = enc.merge_and_unload()          # at export
```

We write it by hand to show the mechanism. It also keeps one name-based wrapper for both encoders, and an export that is plain ModernBERT weights (merged: `W ← W + scale · B·A`) needing no adapter code.

**Head:** the mean of the output token vectors → `Dropout → Linear(H, 256) → GELU → Dropout → Linear(256, 1)`.

**Input:** `<prefix><role>: <message> SEP ... SEP current agent: X SEP question: Q SEP candidate: <name. description>`. The separator and prefix are the encoder's own (ModernBERT: `[SEP]`; Nemotron: a new line and `query: `).

In [ ]:
from transformers import AutoModel, AutoTokenizer

ENCODERS = {
    "modernbert-large": {"name": "answerdotai/ModernBERT-large", "sep": " [SEP] ", "prefix": "", "dtype": torch.float32,
                         "targets": {"Wqkv", "Wo", "Wi"}, "remote": False, "ckpt": False},
    "modernbert-base":  {"name": "answerdotai/ModernBERT-base", "sep": " [SEP] ", "prefix": "", "dtype": torch.float32,
                         "targets": {"Wqkv", "Wo", "Wi"}, "remote": False, "ckpt": False},
    "nemotron-1b":      {"name": "nvidia/llama-nemotron-embed-1b-v2", "sep": "\n", "prefix": "query: ", "dtype": torch.float16,
                         "targets": {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"},
                         "remote": True, "ckpt": True},
}


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update: y = W x + (alpha / r) * B (A x)."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype) @ self.B.t().to(x.dtype)) * self.scale


def add_lora(model, targets):
    found = [(m, n, c) for m in list(model.modules()) for n, c in list(m.named_children()) if n in targets and isinstance(c, nn.Linear)]
    for m, n, c in found:
        setattr(m, n, LoRALinear(c, LORA_R, LORA_ALPHA, LORA_DROPOUT))
    return len(found)


class Model:
    """One encoder with LoRA + one scoring head, and everything needed to score a question."""

    def __init__(self, key):
        self.key, self.spec = key, ENCODERS[key]
        spec = self.spec
        self.tok = AutoTokenizer.from_pretrained(spec["name"], trust_remote_code=spec["remote"])
        self.tok.padding_side, self.tok.truncation_side = "right", "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.enc = AutoModel.from_pretrained(spec["name"], trust_remote_code=spec["remote"], dtype=spec["dtype"]).to(DEVICE)
        self.enc.requires_grad_(False)
        self.n_lora = add_lora(self.enc, spec["targets"])
        assert self.n_lora > 0, f"no LoRA target layers found in {spec['name']}"
        if spec["ckpt"]:
            try:
                self.enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            except Exception as e:                           # some remote-code models do not support it
                log(f"gradient checkpointing not available: {e}")
        self.H = self.enc.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(self.H, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 1)).to(DEVICE)

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode); self.head.train(mode)

    def text(self, item, name):
        sep, desc = self.spec["sep"], item["options"][name]
        parts = [f"{m['role']}: {m['content']}" for m in item["messages"]]
        parts += [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}",
                  f"candidate: {name}. {desc}" if desc else f"candidate: {name}"]
        return self.spec["prefix"] + sep.join(parts)

    def scores(self, items):
        """-> [len(items), K_MAX] scores (padded slots -1e4), one batched encoder pass over every option of every item."""
        inputs, owner, slot = [], [], []
        for b, it in enumerate(items):
            for k, name in enumerate(it["options"]):
                inputs.append(self.text(it, name)); owner.append(b); slot.append(k)
        enc = self.tok(inputs, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        s = self.head((hidden.float() * m).sum(1) / m.sum(1)).squeeze(-1)
        out = torch.full((len(items), K_MAX), -1e4, device=DEVICE)
        out[torch.tensor(owner, device=DEVICE), torch.tensor(slot, device=DEVICE)] = s
        return out

    def probs(self, item):
        """Probabilities over one item's options (in order), no gradient."""
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0, :len(item["options"])].float().cpu()

## 4 — Training and evaluation functions

**Training.** Each epoch shuffles routing batches (`BATCH_ROUTE` questions) and choice batches (`BATCH_CHOICE` questions) together. The loss is listwise cross-entropy over each question's options. AdamW (separate learning rates for adapters and head), 6% warm-up then linear decay, gradient clipping 1.0, fp16 mixed precision. After each epoch both dev sets are scored. The adapters and head with the best **mean** of routing-dev and choice-dev accuracy are kept.

**Evaluation.** Each test question is scored on its own (batch = one question, all its options in one pass), and timed with the GPU synchronized, tokenization included:
- **E10:** live. The model's own previous pick is the current agent, so mistakes can carry over. Reported per turn type, for all 193 turns and for the 83 test-company turns.
- **E11:** first message only, no current agent; top-1 and top-2.
- **Banking:** the last user message of each conversation, with its own current agent.
- **E12:** yes / no, choice, score, per category, the same-text question groups, and the mistakes.

In [ ]:
def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "mean": sum(ms) / len(ms), "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def dev_accuracy(model, items, batch=16):
    model.train(False)
    ok = 0
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            pick = model.scores(chunk).argmax(1).tolist()
            ok += sum(list(it["options"])[k] == it["answer"] for it, k in zip(chunk, pick))
    return ok / max(1, len(items)), ok


def ordinal_loss(scores, items, target):
    """Listwise loss where score questions get an order-aware target: SCORE_EPS of the probability goes to the neighbouring
    levels (near-misses cost less), plus SCORE_DIST_W x |expected level - true level| / (levels - 1) (far misses cost more).
    Yes / no, choice and routing questions keep the plain one-hot target."""
    logp = F.log_softmax(scores, dim=1)
    p = logp.exp()
    soft = torch.zeros_like(scores)
    dist = []
    for b, it in enumerate(items):
        if it["task"] != "score":
            soft[b, target[b]] = 1.0
            continue
        names = list(it["options"])                              # the (shuffled) order of the slots
        level = [it["levels"].index(n) for n in names]           # the scale position of each slot
        true = it["levels"].index(it["answer"])
        nbrs = [k for k, l in enumerate(level) if abs(l - true) == 1]
        soft[b, target[b]] = 1.0 - (SCORE_EPS if nbrs else 0.0)
        for k in nbrs:
            soft[b, k] = SCORE_EPS / len(nbrs)
        expected = (p[b, :len(level)] * torch.tensor(level, dtype=p.dtype, device=p.device)).sum()
        dist.append((expected - true).abs() / (len(level) - 1))
    loss = -(soft * logp).sum(1).mean()
    return loss + SCORE_DIST_W * torch.stack(dist).mean() if dist else loss


class Collapsed(Exception):
    """Training collapsed (run 22: the choice loss stayed at chance after the warm-up peak); retry with another seed."""


COLLAPSE_DEV = {"routing": 0.95, "choice": 0.85}       # healthy runs 19-21 had >= 0.977 / 0.949 after epoch 1


def train_model(model, score_loss="plain", seed=SEED, tasks="both"):
    """tasks: "both" (one model for routing + questions), "routing" or "questions" (a specialist; the other data is not used at all)."""
    use_r, use_c = tasks in ("both", "routing"), tasks in ("both", "questions")
    batches = [train_route[i:i + BATCH_ROUTE] for i in range(0, len(train_route), BATCH_ROUTE)] if use_r else []
    batches += [train_choice[i:i + BATCH_CHOICE] for i in range(0, len(train_choice), BATCH_CHOICE)] if use_c else []
    params = model.lora_params() + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.lora_params(), "lr": LR}, {"params": model.head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
    total = EPOCHS * len(batches)
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, step, t0 = (-1.0, None, 0), 0, time.perf_counter()
    log(f"[{model.key}] score loss: {score_loss}" + (f" (eps {SCORE_EPS}, distance weight {SCORE_DIST_W})" if score_loss == "ordinal" else ""))
    log(f"[{model.key}] training ({tasks}): {len(batches)} batches per epoch ({len(train_route) if use_r else 0} routing + "
        f"{len(train_choice) if use_c else 0} choice questions), {total} steps")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        order = list(range(len(batches)))
        random.Random(seed + epoch).shuffle(order)
        run = {"routing": [], "choice": []}
        for bi in order:
            batch = batches[bi]
            items = [dict(it) for it in batch]
            for it in items:                                    # option order shuffled, so position can't be learned
                names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            target = torch.tensor([list(it["options"]).index(it["answer"]) for it in items], device=DEVICE)
            out = model.scores(items)
            loss = ordinal_loss(out, items, target) if score_loss == "ordinal" else F.cross_entropy(out, target)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run["routing" if batch[0]["task"] == "routing" else "choice"].append(loss.item())
            step += 1
            if step % 100 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  [{model.key}] epoch {epoch} step {step}/{total} | loss routing {statistics.mean(run['routing'][-50:] or [0]):.4f} "
                    f"choice {statistics.mean(run['choice'][-50:] or [0]):.4f} | {step / el:.2f} steps/s | "
                    f"~{(total - step) / (step / el) / 60:.1f} min left | {gpu()}")
        accs, msg = [], []
        if use_r:
            acc_r, ok_r = dev_accuracy(model, dev_route); accs.append(acc_r); msg.append(f"dev routing {ok_r}/{len(dev_route)} ({acc_r:.1%})")
        if use_c:
            acc_c, ok_c = dev_accuracy(model, dev_choice); accs.append(acc_c); msg.append(f"dev choice {ok_c}/{len(dev_choice)} ({acc_c:.1%})")
        log(f"[{model.key}] epoch {epoch} done | " + " | ".join(msg))
        if epoch == 1 and ((use_r and acc_r < COLLAPSE_DEV["routing"]) or (use_c and acc_c < COLLAPSE_DEV["choice"])):
            raise Collapsed(" | ".join(msg) + f" after epoch 1 (thresholds {COLLAPSE_DEV})")
        if sum(accs) / len(accs) > best[0]:
            best = (sum(accs) / len(accs), {"lora": {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad},
                                          "head": copy.deepcopy(model.head.state_dict())}, epoch)
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[1]["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1]["head"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"[{model.key}] training done in {minutes:.1f} min | best epoch {best[2]} (mean dev accuracy {best[0]:.1%})")
    return {"train_minutes": minutes, "best_epoch": best[2]}


TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]


def eval_e10(model):
    model.train(False)
    rows, lat = [], []
    for conv in real_convs:
        cur = None
        for u in conv["user_turns"]:
            prefix = conv["messages"][:u["message_index"] + 1]
            item = {"messages": prefix, "current": cur, "question": ROUTE_Q, "options": conv["agents"]}
            p, ms = timed_probs(model, item)
            lat.append(ms)
            cur = list(conv["agents"])[int(p.argmax())]
            rows.append({"company": conv["company"], "type": u["turn_type"], "ok": cur == u["expected_agent"]})
    test = [r for r in rows if r["company"] in TEST_COMPANIES]
    print(f"E10 live routing | all {sum(r['ok'] for r in rows)}/{len(rows)} | test companies {sum(r['ok'] for r in test)}/{len(test)}")
    print(f"  {'turn type':14s} | {'all':>9} | {'test':>9}")
    for tt in TURN_TYPES:
        a, t_ = [r for r in rows if r["type"] == tt], [r for r in test if r["type"] == tt]
        if a:
            print(f"  {tt:14s} | {sum(r['ok'] for r in a):>3} / {len(a):<4} | {sum(r['ok'] for r in t_):>3} / {len(t_):<4}")
    return {"all": sum(r["ok"] for r in rows), "all_n": len(rows), "test": sum(r["ok"] for r in test), "test_n": len(test),
            "latency": lat_summary(lat)}


def eval_e11(model):
    model.train(False)
    top1 = top2 = 0
    lat = []
    for m in ft["messages"]:
        agents = ft["companies"][m["company"]]
        item = {"messages": [{"role": "user", "content": m["message"]}], "current": None, "question": ROUTE_Q, "options": agents}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        names = list(agents)
        top1 += names[int(p.argmax())] == m["target_agent"]
        top2 += m["target_agent"] in [names[i] for i in p.topk(2).indices.tolist()]
    print(f"E11 first turn | top-1 {top1}/{len(ft['messages'])} | top-2 {top2}/{len(ft['messages'])}")
    return {"top1": top1, "top2": top2, "n": len(ft["messages"]), "latency": lat_summary(lat)}


def eval_banking(model):
    model.train(False)
    ok, lat = [], []
    for ex in banking:
        item = {"messages": ex["messages"], "current": ex["current_agent"], "question": ROUTE_Q, "options": BANKING_AGENTS}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        ok.append(list(BANKING_AGENTS)[int(p.argmax())] == ex["target_agent"])
    bench = sum(ok[:50])
    print(f"banking | benchmark {bench}/{min(50, len(ok))} | all {sum(ok)}/{len(ok)}")
    return {"benchmark": bench, "all": sum(ok), "n": len(ok), "latency": lat_summary(lat)}


def eval_e12(model):
    model.train(False)
    rows, lat = [], []
    for it in e12:
        item = {"messages": [{"role": "user", "content": it["state"]}], "current": None, "question": it["question"],
                "options": {o: "" for o in it["options"]}}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        top = int(p.argmax())
        row = {**it, "pick": it["options"][top], "ok": it["options"][top] == it["answer"], "probs": p.tolist()}
        if it["type"] == "score":
            row["level_error"] = abs(top - it["answer_index"])
            row["expected_level"] = float(sum(i * pi for i, pi in enumerate(p.tolist())))
        rows.append(row)
    by = {t: sum(r["ok"] for r in rows if r["type"] == t) for t in ["noul", "choice", "score"]}
    sc = [r for r in rows if r["type"] == "score"]
    groups = {}
    for r in rows:
        if "pair" in r:
            groups.setdefault(r["pair"], []).append(r)
    g_ok = sum(all(r["ok"] for r in g) for g in groups.values())
    exp_ok = sum(min(len(r["options"]) - 1, max(0, round(r["expected_level"]))) == r["answer_index"] for r in sc)
    print(f"E12 choice | all {sum(r['ok'] for r in rows)}/{len(rows)} | yes/no {by['noul']} | choice {by['choice']} | score {by['score']} "
          f"(within 1 level {sum(r['level_error'] <= 1 for r in sc)}/{len(sc)}) | question groups {g_ok}/{len(groups)} | "
          f"answered yes {sum(r['pick'] == 'yes' for r in rows if r['type'] == 'noul')} (truth {sum(r['answer'] == 'yes' for r in rows if r['type'] == 'noul')})")
    print(f"  score with the rounded expected level instead of the top level: {exp_ok}/{len(sc)} exact")
    print("  per category: " + ", ".join(f"{c} {sum(r['ok'] for r in rows if r['category'] == c)}/{sum(r['category'] == c for r in rows)}"
                                         for c in sorted({r['category'] for r in rows})))
    print("  mistakes:")
    for r in [r for r in rows if not r["ok"]]:
        s_ = r["state"] if len(r["state"]) <= 44 else r["state"][:41] + "..."
        print(f"    {r['id']} {r['type']:6s} | {s_:44s} | {r['question'][:36]:36s} | expected {r['answer']:14s} | got {r['pick']} ({max(r['probs']):.2f})")
    return {"all": sum(r["ok"] for r in rows), "n": len(rows), **by, "groups": g_ok, "score_expected_rule": exp_ok,
            "score_within1": sum(r["level_error"] <= 1 for r in sc), "latency": lat_summary(lat)}


def eval_ind_questions(model):
    """Independent questions: yes / no, choice, score; per type, yes / no by truth, per question wording."""
    model.train(False)
    rows, lat = [], []
    for it in ind_q:
        item = {"messages": [{"role": "user", "content": it["state"]}], "current": None, "question": it["question"],
                "options": {o: "" for o in it["options"]}}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        top = int(p.argmax())
        rows.append({**it, "pick": it["options"][top], "ok": it["options"][top] == it["answer"],
                     "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
    by = {t: sum(r["ok"] for r in rows if r["type"] == t) for t in ["noul", "choice", "score"]}
    yn = [r for r in rows if r["type"] == "noul"]
    yes_ok = sum(r["ok"] for r in yn if r["answer"] == "yes"); n_yes = sum(r["answer"] == "yes" for r in yn)
    sc = [r for r in rows if r["type"] == "score"]
    print(f"IND questions | all {sum(r['ok'] for r in rows)}/{len(rows)} | yes/no {by['noul']}/{len(yn)} "
          f"(truth yes: {yes_ok}/{n_yes}, truth no: {by['noul'] - yes_ok}/{len(yn) - n_yes}) | choice {by['choice']}/{sum(r['type'] == 'choice' for r in rows)} "
          f"| score {by['score']}/{len(sc)} (within 1 level {sum(r['level_error'] <= 1 for r in sc)})")
    clean = [r for r in rows if not r.get("flag")]
    claim = [r for r in rows if r.get("flag") == "claim_check"]
    sc_ok = [r for r in sc if r.get("flag") != "unordered_scale"]
    print(f"  without flagged items: {sum(r['ok'] for r in clean)}/{len(clean)} | claim-check yes/no (all truth 'yes'): "
          f"{sum(r['ok'] for r in claim)}/{len(claim)} | score on proper scales: {sum(r['ok'] for r in sc_ok)}/{len(sc_ok)} "
          f"(within 1 level {sum(r['level_error'] <= 1 for r in sc_ok)}) | unordered 'scales': "
          f"{sum(r['ok'] for r in sc if r.get('flag') == 'unordered_scale')}/{len(sc) - len(sc_ok)}")
    return {"all": sum(r["ok"] for r in rows), "n": len(rows), **by, "yes_truth": yes_ok, "no_truth": by["noul"] - yes_ok,
            "unflagged": sum(r["ok"] for r in clean), "n_unflagged": len(clean), "score_proper": sum(r["ok"] for r in sc_ok),
            "latency": lat_summary(lat)}


def eval_ind_routing(model):
    """Independent conversations, two ways: LIVE (the model's previous pick is the current agent) and GIVEN (the correct
    previous agent is the current agent, so one arbitrary label cannot cascade). Reported for all turns and for the
    turns whose label is not flagged (see build_independent_test_data.py)."""
    model.train(False)
    res, lat = {}, []
    for mode in ["live", "given"]:
        rows = []
        for conv in ind_r:
            cur, prev_true = None, None
            for u in conv["user_turns"]:
                prefix = conv["messages"][:u["message_index"] + 1]
                current = cur if mode == "live" else prev_true
                item = {"messages": prefix, "current": current, "question": ROUTE_Q, "options": conv["agents"]}
                p, ms = timed_probs(model, item)
                if mode == "live":
                    lat.append(ms)
                cur = list(conv["agents"])[int(p.argmax())]
                rows.append({"type": u["turn_type"], "flag": u["flag"], "ok": cur == u["expected_agent"]})
                prev_true = u["expected_agent"]
        clean = [r for r in rows if not r["flag"]]
        res[mode] = {"all": sum(r["ok"] for r in rows), "n": len(rows), "unflagged": sum(r["ok"] for r in clean), "n_unflagged": len(clean)}
        print(f"IND routing ({mode}) | all {res[mode]['all']}/{len(rows)} | unflagged turns {res[mode]['unflagged']}/{len(clean)} | "
              + " ".join(f"{t} {sum(r['ok'] for r in rows if r['type'] == t)}/{sum(r['type'] == t for r in rows)}" for t in TURN_TYPES
                         if any(r["type"] == t for r in rows))
              + " | flagged: " + " ".join(f"{f} {sum(r['ok'] for r in rows if r['flag'] == f)}/{sum(r['flag'] == f for r in rows)}"
                                         for f in ["vague_start", "label_conflict"]))
    res["latency"] = lat_summary(lat)
    return res


def eval_ind_first(model):
    model.train(False)
    rows, lat = [], []
    for m in ind_f:
        item = {"messages": [{"role": "user", "content": m["message"]}], "current": None, "question": ROUTE_Q, "options": m["agents"]}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        names = list(m["agents"])
        rows.append({"type": m["type"], "ok": names[int(p.argmax())] == m["target_agent"],
                     "top2": m["target_agent"] in [names[i] for i in p.topk(2).indices.tolist()]})
    print(f"IND first turn | top-1 {sum(r['ok'] for r in rows)}/{len(rows)} | top-2 {sum(r['top2'] for r in rows)}/{len(rows)} | "
          + " ".join(f"{k} {sum(r['ok'] for r in rows if r['type'] == k)}/{sum(r['type'] == k for r in rows)}" for k in sorted({r['type'] for r in rows})))
    return {"top1": sum(r["ok"] for r in rows), "top2": sum(r["top2"] for r in rows), "n": len(rows), "latency": lat_summary(lat)}


ROUTING_TESTS = [("E10", "eval_e10"), ("E11", "eval_e11"), ("banking", "eval_banking"), ("IND routing", "eval_ind_routing"),
                 ("IND first turn", "eval_ind_first")]
QUESTION_TESTS = [("E12", "eval_e12"), ("IND questions", "eval_ind_questions")]


def evaluate_all(model, tasks="both"):
    """Each model is scored only on the tests of the job it was trained for."""
    model.train(False)
    model.probs({"messages": [{"role": "user", "content": "warm-up"}], "current": None, "question": ROUTE_Q, "options": BANKING_AGENTS})
    res = {}
    tests = (ROUTING_TESTS if tasks in ("both", "routing") else []) + (QUESTION_TESTS if tasks in ("both", "questions") else [])
    for name, fn in [(n, globals()[f]) for n, f in tests]:
        log(f"[{model.key}] evaluating {name}")
        res[name] = fn(model)
        L = res[name]["latency"]
        print(f"  latency {name}: p50 {L['p50']:.0f} ms | p95 {L['p95']:.0f} ms | mean {L['mean']:.0f} ms ({L['n']} questions)\n")
    return res


def merge_lora(enc):
    """Fold every LoRA update into its frozen weight (W <- W + scale * B A) and put the plain linear layer back."""
    n = 0
    for module in list(enc.modules()):
        for name, child in list(module.named_children()):
            if isinstance(child, LoRALinear):
                with torch.no_grad():
                    child.base.weight += ((child.B.float() @ child.A.float()) * child.scale).to(child.base.weight.dtype)
                setattr(module, name, child.base)
                n += 1
    return n


def export_signpost(model, res, out=Path("signpost")):
    """Save the trained model as a self-contained folder (merged encoder in fp16, tokenizer, head, config, signpost.py),
    zip it, then reload it with signpost.py and check it reproduces this run's E11 and E12 scores."""
    import shutil, sys
    if out.exists():
        shutil.rmtree(out)
    (out / "encoder").mkdir(parents=True)
    n = merge_lora(model.enc)
    model.enc.half().save_pretrained(out / "encoder", safe_serialization=True)
    model.tok.save_pretrained(out / "encoder")
    torch.save({"state_dict": {k: v.float().cpu() for k, v in model.head.state_dict().items()}, "hidden": model.H}, out / "head.pt")
    cfg = {"name": "Signpost", "base_model": model.spec["name"], "license_of_base": "Apache-2.0", "sep": model.spec["sep"],
           "prefix": model.spec["prefix"], "max_len": MAX_LEN, "route_question": ROUTE_Q, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
           "score_loss": res["score_loss"], "epochs": EPOCHS, "best_epoch": res.get("best_epoch"), "seed_attempts": res.get("attempts"),
           "trained_on": {"routing": len(train_route), "questions": len(train_choice)}, "data_branch": DATA_BRANCH,
           "results": {"E10_test": res["E10"]["test"], "E10_all": res["E10"]["all"], "E11_top1": res["E11"]["top1"],
                       "banking_50": res["banking"]["benchmark"], "banking_225": res["banking"]["all"], "E12": res["E12"]["all"],
                       "IND_questions": res["IND questions"]["all"], "IND_routing_live": res["IND routing"]["live"]["all"],
                       "IND_first_turn": res["IND first turn"]["top1"]}}
    json.dump(cfg, open(out / "signpost_config.json", "w"), indent=1)
    shutil.copy(DATA_DIR.parent.parent / "signpost" / "signpost.py", out / "signpost.py")
    shutil.make_archive("signpost", "zip", root_dir=".", base_dir=str(out))
    size = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1e9
    log(f"exported {out}/ ({n} LoRA layers merged, {size:.2f} GB) and signpost.zip ({Path('signpost.zip').stat().st_size / 1e9:.2f} GB)")
    # ---- verify: reload from disk with signpost.py, score E11 and E12 again
    sys.path.insert(0, str(out.resolve()))
    from signpost import Signpost
    check = {}
    for dtype in [torch.float16, torch.float32]:              # fp16 = GPU use; fp32 = what a CPU run computes
        sp = Signpost(out, device=DEVICE, dtype=dtype)
        n_e11 = sum(sp.route([{"role": "user", "content": m["message"]}], ft["companies"][m["company"]])[0] == m["target_agent"]
                    for m in ft["messages"])
        n_e12 = sum(sp.choice(it["state"], it["question"], it["options"])[0] == it["answer"] for it in e12)
        check[str(dtype)] = {"E11": n_e11, "E12": n_e12}
        log(f"export check ({dtype}): E11 {n_e11}/{len(ft['messages'])} (trained model {res['E11']['top1']}) | "
            f"E12 {n_e12}/{len(e12)} (trained model {res['E12']['all']})")
        del sp
    return {"folder": str(out), "zip": "signpost.zip", "gb": round(size, 2), "check": check}


def run_encoder(key, score_loss="plain", max_attempts=3, tasks="both"):
    """Build, train and evaluate one encoder with one score loss; free the GPU afterwards. -> results dict (also saved to JSON).
    If training collapses (see COLLAPSE_DEV), it restarts from scratch with the next seed; every attempt is recorded."""
    log(f"===== {key} / {score_loss} score loss / trained on: {tasks} =====")
    attempts = []
    for attempt in range(max_attempts):
        seed = SEED + 1000 * attempt
        random.seed(seed); torch.manual_seed(seed)              # attempt 0 = the usual seed, as in runs 19-21
        if AMP:
            torch.cuda.reset_peak_memory_stats(DEVICE)
        model = Model(key)
        log(f"[{key}] attempt {attempt + 1} (seed {seed}) | {model.spec['name']}: {model.n_lora} LoRA layers | trainable "
            f"{sum(p.numel() for p in model.lora_params()) / 1e6:.1f} M + head {sum(p.numel() for p in model.head.parameters()) / 1e6:.2f} M | {gpu()}")
        try:
            trained = train_model(model, score_loss, seed, tasks)
            attempts.append({"seed": seed, "status": "ok"})
            break
        except Collapsed as e:
            log(f"[{key}] attempt {attempt + 1} COLLAPSED: {e} -> restarting with a new seed")
            attempts.append({"seed": seed, "status": f"collapsed: {e}"})
            del model; gc.collect()
            if AMP:
                torch.cuda.empty_cache()
    else:
        raise RuntimeError(f"{key}: training collapsed in all {max_attempts} attempts: {attempts}")
    res = {"encoder": key, "score_loss": score_loss, "tasks": tasks, "attempts": attempts, **trained}
    res.update(evaluate_all(model, tasks))
    res["peak_gpu_gb"] = torch.cuda.max_memory_allocated(DEVICE) / 1e9 if AMP else 0
    if EXPORT and tasks == "both":
        res["export"] = export_signpost(model, res)
    with open(f"lora_signpost_{key}_{score_loss}_{tasks}.json", "w") as fh:
        json.dump(res, fh, indent=1)
    del model
    gc.collect()
    if AMP:
        torch.cuda.empty_cache()
    return res


RESULTS = {}

## 5 — Train Signpost and export it

`TRAIN_MODE = "combined"`: one ModernBERT-large model for routing and questions, with the final recipe:
- LoRA r 16 and a scoring head;
- ordinal score loss;
- 2 epochs, seed 42;
- collapse guard;
- the same data.

After the tests, `EXPORT = 1` does four things:
1. **merges** the LoRA updates into the encoder weights (`W ← W + scale · B·A`), so the model needs no adapter code;
2. saves **`signpost/`**: `encoder/` (fp16 safetensors, config and tokenizer), `head.pt`, `signpost_config.json` (input format, settings, this run's scores) and `signpost.py` (the standalone loader: `route`, `yes_no`, `choice`, `score`);
3. zips it to **`signpost.zip`** in `/kaggle/working`;
4. **reloads the folder with `signpost.py`** and scores E11 and E12 again, in fp16 (GPU) and fp32 (what a CPU run computes). The numbers must match the trained model's within a point or so.

**Expect.** The training and test output as before, then the export lines and the two check lines.


In [ ]:
if RUN_ENCODERS[0] in ENCODERS:
    for loss_name in SCORE_LOSSES:
        if TRAIN_MODE == "separate":                            # two independent specialists, same recipe and seed (run 24)
            RESULTS[f"{RUN_ENCODERS[0]} / routing only"] = run_encoder(RUN_ENCODERS[0], loss_name, tasks="routing")
            RESULTS[f"{RUN_ENCODERS[0]} / questions only"] = run_encoder(RUN_ENCODERS[0], loss_name, tasks="questions")
        else:
            RESULTS[f"{RUN_ENCODERS[0]} / {loss_name}"] = run_encoder(RUN_ENCODERS[0], loss_name)

## 6 — Optional: a second encoder (Nemotron-1B)

Runs only if `RUN_ENCODERS` lists a second encoder (e.g. `modernbert-large,nemotron-1b`, as in run 19): the same data, settings and tests. Nemotron is about three times larger, so training is slower (gradient checkpointing on). If it runs out of GPU memory, set a smaller `BATCH_ROUTE` (e.g. 2), re-execute Section 1 and rerun this cell; the results above are kept.

In [ ]:
for key in RUN_ENCODERS[1:]:
    for loss_name in SCORE_LOSSES:
        RESULTS[f"{key} / {loss_name}"] = run_encoder(key, loss_name)

## 7 — Side by side: this run, the frozen system and the decider

Every column is scored on the same test items. The reference numbers come from the experiment ledger: the frozen system is `02_frozen_router.ipynb` (run 17, Nemotron router alone on E10), and the Strands decider comes from `04_decider_benchmark.ipynb`. The decider has no current-agent input; on conversations it is shown with the full history (as our model sees it) and, separately, with only the last 3 user turns, its best setting. Its latency is in `04`, Section 11.

In [ ]:
REF = {
    "frozen system (02, run 17)": {"E10 test (83)": 69, "E10 all (193)": 158, "E11 top-1 (100)": 94, "banking 50": 43, "banking 225": 194,
                                   "E12 (170)": 114, "E12 yes/no (58)": 38, "E12 choice (59)": 44, "E12 score (53)": 32},
    "decider 2B, full history": {"E10 test (83)": 62, "E10 all (193)": 151, "E11 top-1 (100)": 96, "banking 50": 48, "banking 225": 214,
                                 "E12 (170)": 160, "E12 yes/no (58)": 56, "E12 choice (59)": 56, "E12 score (53)": 48,
                                 "IND questions (300)": 253, "IND questions unflagged (278)": 253, "IND yes/no (100)": 77,
                                 "IND choice (108)": 106, "IND score (92)": 70, "IND routing live (172)": 113,
                                 "IND routing live, unflagged (171)": 112, "IND first turn (200)": 178},
    "decider 2B, last 3 turns": {"E10 test (83)": 69, "E10 all (193)": 163, "IND routing live (172)": 121, "IND routing live, unflagged (171)": 120},
}
rows = ["E10 test (83)", "E10 all (193)", "E11 top-1 (100)", "banking 50", "banking 225", "E12 (170)",
        "E12 yes/no (58)", "E12 choice (59)", "E12 score (53)",
        "IND questions (300)", "IND questions unflagged (278)", "IND yes/no (100)", "IND choice (108)", "IND score (92)",
        "IND routing live (172)", "IND routing live, unflagged (171)", "IND routing given prev (172)", "IND first turn (200)"]

def ours(r, row):
    try:
        return _ours(r, row)
    except KeyError:                                            # a specialist is not scored on the other job's tests
        return "—"


def _ours(r, row):
    return {"E10 test (83)": lambda r: r["E10"]["test"], "E10 all (193)": lambda r: r["E10"]["all"], "E11 top-1 (100)": lambda r: r["E11"]["top1"],
            "banking 50": lambda r: r["banking"]["benchmark"], "banking 225": lambda r: r["banking"]["all"], "E12 (170)": lambda r: r["E12"]["all"],
            "E12 yes/no (58)": lambda r: r["E12"]["noul"], "E12 choice (59)": lambda r: r["E12"]["choice"], "E12 score (53)": lambda r: r["E12"]["score"],
            "IND questions (300)": lambda r: r["IND questions"]["all"], "IND yes/no (100)": lambda r: r["IND questions"]["noul"],
            "IND choice (108)": lambda r: r["IND questions"]["choice"], "IND score (92)": lambda r: r["IND questions"]["score"],
            "IND questions unflagged (278)": lambda r: r["IND questions"]["unflagged"],
            "IND routing live (172)": lambda r: r["IND routing"]["live"]["all"], "IND routing live, unflagged (171)": lambda r: r["IND routing"]["live"]["unflagged"],
            "IND routing given prev (172)": lambda r: r["IND routing"]["given"]["all"], "IND first turn (200)": lambda r: r["IND first turn"]["top1"]}[row](r)


cols = [f"LoRA {k}" for k in RESULTS] + list(REF)
print(f"{'accuracy':18s} | " + " | ".join(f"{c:>26s}" for c in cols))
for row in rows:
    vals = [ours(RESULTS[k], row) for k in RESULTS] + [REF[c].get(row, "—") for c in REF]
    print(f"{row:18s} | " + " | ".join(f"{str(v):>26s}" for v in vals))

print(f"\n{'latency (ms)':18s} | " + " | ".join(f"{('LoRA ' + k):>26s}" for k in RESULTS))
for test in ["E10", "E11", "banking", "E12", "IND questions", "IND routing", "IND first turn"]:
    print(f"{test + ' p50 / p95':18s} | " + " | ".join((f"{RESULTS[k][test]['latency']['p50']:>15.0f} / {RESULTS[k][test]['latency']['p95']:<8.0f}"
                                                         if test in RESULTS[k] else f"{'—':>26s}") for k in RESULTS))
print(f"{'training (min)':18s} | " + " | ".join(f"{RESULTS[k]['train_minutes']:>26.1f}" for k in RESULTS))
print(f"{'peak GPU (GB)':18s} | " + " | ".join(f"{RESULTS[k]['peak_gpu_gb']:>26.1f}" for k in RESULTS))

---
## 8 — What we tried and dropped

Every number is correct / total on the same test items; the run numbers point to `experiments/modernbert_router_ledger.md`. One seed per run. Three runs of the final recipe (21, 22b, 25) put the **run-to-run noise at about ±4** on most tests, so smaller differences aren't evidence.

| Run | What we tried | Result | Decision |
|---|---|---|---|
| 18 | LoRA + head, **grounded questions only** (no routing) | E12 **114 → 153** / 170 (decider 160) in 9 minutes of training; score answers never more than one level off | The step change of the project: the plateau was the frozen encoder. Kept, and extended to routing |
| 19 | **One model for both jobs**, ModernBERT-large vs Nemotron-1B | Both beat the frozen system and the decider on every routing test with no detector or gate (E10 test 75 and 79 / 83 vs 62). ModernBERT kept E12 at 153 while also routing; Nemotron scored 141 | One combined model. The encoders differ: Nemotron routes better, ModernBERT reads better (Nemotron misread numbers, e.g. "45 cm wide" → over 2 m at 0.99) |
| 20 | **More score training data** (709 → 2,004 items) | E12 score stayed at 43 / 53; urgency got *worse* (8 vs 10 / 13) | Dropped as a lever. A new deadline-urgency family taught a shortcut, "no deadline word → low urgency", and E12's urgency comes from severity ("payouts failing") |
| 21 | **Clean data** (training texts that were near-paraphrases of E12 items reworded, and a guard added) + severity-based urgency, then **plain vs ordinal** score loss | Plain: E12 155, score 46. Ordinal: **E12 161** (decider 160), score 47, E10 test 81 / 83 | Ordinal loss adopted (never worse on score, keeps answers within one level). The guard stays. Routing also moved by 4 between the two variants although the loss only touches score questions: the first measure of fine-tuning noise |
| 22 | The same recipe, rerun | **Training collapsed**: the choice loss stayed at chance after the learning-rate peak; E12 88, dev choice 599 / 1,014 | A **collapse guard**: after epoch 1, if dev routing < 95% or dev choice < 85%, restart with the next seed (up to 3 attempts). The rerun (22b) was healthy and reproduced run 21 within noise (E10 test 80, E12 161) |
| 23 | **Nemotron-1B** on the final recipe, 3 epochs | Routing tied ModernBERT (independent live routing 134 vs 131); questions clearly worse (independent 200 vs 231, score 34 vs 53 / 92); 3× the parameters and training time | ModernBERT-large stays the final encoder |
| 24 | **Two specialists** (routing only, questions only) vs one combined model | No specialist beat the combined model beyond noise; the two biggest gaps favoured the combined one (banking 210 vs 218 / 225, E12 155 vs 161) | **One model.** Joint training doesn't pollute either job and may help both |

**Also dropped once the encoder was fine-tuned:** the context-change detector and the confident-stay gate from `02`. The fine-tuned router handles stays and switches on its own (E10 real switches 20 / 20 in run 21).

**What the independent test showed** (written outside the project, run 22b and 25):
- **Routing holds up:** live routing 133 vs 113 / 172 for the decider (121 with a 3-turn window), mostly on switches.
- **Score precision does not:** independent score 43–53 / 92 across seeds, against the decider's 70. E12's score parity came from test items written in the same style as our training data.
- **Our model switches too readily** on stay-type turns written by someone else (37 vs 48 / 55 for the decider in run 22b).

By the independence rule, none of this was fed back into training. These are the published model's known limits, and they're stated on its model card.

---
## 9 — Next

The exported folder is on the Hugging Face Hub as **[skamalj/signpost](https://huggingface.co/skamalj/signpost)**, with the standalone loader `signpost.py` (`route`, `yes_no`, `choice`, `score`). The model card's examples were run on the published files. `04_decider_benchmark.ipynb` produces the decider's side of every comparison above.

**Part 5** (branch `signpost-apps`) uses Signpost as the router of a real multi-agent app: LangGraph with five Claude agents, a DynamoDB checkpointer and a message reducer.